# Titanic Exploratory Data Analysis

This notebook explores the Titanic dataset to identify patterns, trends, and anomalies related to survival.

## Goals
- Check dataset shape, missing values, and variable types
- Investigate univariate and bivariate relationships
- Highlight strong survival patterns by sex, class, and fare
- Summarize actionable insights from the visual analysis

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

train = pd.read_csv(r'c:\Users\acer\Downloads\Task-5-Exploratory-Data-Analysis\train.csv')
train.head()

In [ ]:
train.info()
train.isnull().sum()

In [ ]:
train.describe(include='all').T

## Observation 1: Data quality and missing values
- There are 891 entries and 12 columns.
- Age has 177 missing values and Cabin has a large number of missing values, so it should be treated carefully in any model.
- Embarked has only 2 missing entries, so we can fill them with the mode.

In [ ]:
train['Age'] = train['Age'].fillna(train['Age'].median())
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['HasCabin'] = train['Cabin'].notna().astype(int)

train[['Age', 'Embarked', 'FamilySize', 'HasCabin']].head()

## Observation 2: Survival is strongly associated with sex and travel class
Women had a much higher survival rate than men, and first-class passengers survived more often than passengers in lower classes.

In [ ]:
survival_rate = train['Survived'].mean()
sex_survival = train.groupby('Sex')['Survived'].mean().sort_values(ascending=False)
class_survival = train.groupby('Pclass')['Survived'].mean().sort_values(ascending=False)
embarked_survival = train.groupby('Embarked')['Survived'].mean().sort_values(ascending=False)

print(f'Survival rate overall: {survival_rate:.2%}')
print("Survival by sex:
", sex_survival)
print("Survival by class:
", class_survival)
print("Survival by embarkation port:
", embarked_survival)

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(x=sex_survival.index, y=sex_survival.values, ax=axes[0])
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival rate')

sns.barplot(x=class_survival.index, y=class_survival.values, ax=axes[1])
axes[1].set_title('Survival Rate by Pclass')
axes[1].set_ylabel('Survival rate')

sns.barplot(x=embarked_survival.index, y=embarked_survival.values, ax=axes[2])
axes[2].set_title('Survival Rate by Embarked Port')
axes[2].set_ylabel('Survival rate')

plt.tight_layout()
plt.show()

## Observation 3: Age and fare distributions reveal wealth and age-related differences
Passengers who survived tended to be younger and had higher fares, which suggests a connection between passenger class and financial status.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.histplot(train['Age'], bins=30, kde=True, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Age Distribution')

sns.boxplot(x='Survived', y='Age', data=train, ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Age by Survival')

sns.histplot(train['Fare'], bins=30, kde=True, ax=axes[1,0], color='darkorange')
axes[1,0].set_title('Fare Distribution')

sns.boxplot(x='Survived', y='Fare', data=train, ax=axes[1,1], palette='pastel')
axes[1,1].set_title('Fare by Survival')

plt.tight_layout()
plt.show()

## Observation 4: Correlation and pairwise relationships
The strongest numeric relationships are found between passenger class, fare, and survival, while family size and Parch/SibSp show moderate association.

In [ ]:
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
correlation = train[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', center=0, linewidths=.5, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
pair_data = train[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']].dropna()
sns.pairplot(pair_data, hue='Survived', palette='husl', diag_kind='hist', plot_kws={'alpha': 0.7})
plt.show()

## Final insights
1. Sex was the strongest factor for survival: women survived much more often than men.
2. Higher-class passengers had more access to lifeboats and better cabins, which translated to higher survival rates.
3. Fare was positively associated with survival, confirming the class effect.
4. Age was a secondary factor; younger passengers and children were more likely to survive than older adults.
5. Missing data in Age and Cabin should be handled carefully during modeling, but the overall dataset still clearly shows actionable patterns.